# Tuning kernels inside a larger program with `jax.experimental.rebindable`

A kernel's *hyperparameters* (tile sizes, chunk sizes, number of stages, ...)
change how fast it runs but never what it computes. The best values depend on
the shapes the kernel actually sees, which are often only known deep inside a
larger program, e.g. after a `shard_map` splits the batch or after a model
reshapes its activations.

`jax.experimental.rebindable` lets you:

1. **declare** a kernel's hyperparameters with `rebindable(fn, hyperparams=...)`,
2. **trace** the whole program once with `jax.jit(f).trace(...)`,
3. **find** every kernel call (a `RebindableSite`) with `extract_rebindables`,
   together with the exact operand types it sees,
4. **benchmark** each site *on its own* with `site.call(...)`,
5. **apply** the winners with `rebind(traced, ...)` and lower the program,
   without retracing it (only the rebound kernel bodies are retraced).

This notebook runs on CPU. The "kernels" are plain JAX functions whose
hyperparameter is a chunk size, which already has a visible effect on CPU
runtime; on an accelerator you would do exactly the same with Pallas kernels.

In [1]:
import os
# 4 fake CPU devices for the sharded section below; must be set before jax is imported.
os.environ["XLA_FLAGS"] = (os.environ.get("XLA_FLAGS", "")
                           + " --xla_force_host_platform_device_count=4")

import time
import itertools
from functools import partial

import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental.rebindable import (
    rebindable, extract_rebindables, rebind, rebinding)

print(jax.__version__, jax.devices())

0.12.0.dev20261009+164c9caab7 [CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3)]


## 1. Declaring rebindable kernels

Two toy kernels, each with one hyperparameter:

* `chunked_attention` processes queries `block_q` at a time (like a
  flash-attention kernel's query tile),
* `chunked_mlp` processes tokens `block_m` rows at a time.

Hyperparameters are keyword arguments, static like `jax.jit`'s
`static_argnames` (other static arguments can be marked with
`static_argnums`/`static_argnames`; everything else may be an array). They
must be hashable and must not change output types. `rebindable_id=` is an
optional label we can use to rebind sites from a table (a call may override it
with its own `rebindable_id=`).

In [2]:
@partial(rebindable, hyperparams=("block_q",), rebindable_id="attention")
def chunked_attention(q, k, v, *, block_q=8):
  T, D = q.shape
  qb = q.reshape(T // block_q, block_q, D)

  def one_block(qi):
    s = qi @ k.T / jnp.sqrt(D).astype(q.dtype)
    return jax.nn.softmax(s, axis=-1) @ v

  return jax.lax.map(one_block, qb).reshape(T, D)


@partial(rebindable, hyperparams=("block_m",), rebindable_id="mlp")
def chunked_mlp(x, w1, w2, *, block_m=8):
  T, D = x.shape
  xb = x.reshape(T // block_m, block_m, D)
  return jax.lax.map(lambda xi: jax.nn.gelu(xi @ w1) @ w2, xb).reshape(T, -1)

## 2. A larger program

A small transformer-like model: a few layers, each calling both kernels, all
under `vmap` over the batch. Nothing in the model code knows about tuning.

In [3]:
D, H, T, B, L = 64, 256, 512, 4, 3

def init_params(key):
  ks = jax.random.split(key, 4 * L)
  n = lambda k, s: jax.random.normal(k, s, jnp.float32) / np.sqrt(s[0])
  return [dict(wq=n(ks[4*i], (D, D)), wkv=n(ks[4*i+1], (D, 2 * D)),
               w1=n(ks[4*i+2], (D, H)), w2=n(ks[4*i+3], (H, D)))
          for i in range(L)]

def layer(p, x):  # x: [T, D] (one sequence)
  q = x @ p["wq"]
  k, v = jnp.split(x @ p["wkv"], 2, axis=-1)
  x = x + chunked_attention(q, k, v, block_q=8)
  return x + chunked_mlp(jax.nn.standardize(x), p["w1"], p["w2"], block_m=8)

def model(params, xs):  # xs: [B, T, D]
  def one(x):
    for p in params:
      x = layer(p, x)
    return x
  return jax.vmap(one)(xs)

params = init_params(jax.random.key(0))
xs = jax.random.normal(jax.random.key(1), (B, T, D), jnp.float32)
model_jit = jax.jit(model)
print(model_jit(params, xs).shape)

(4, 512, 64)


## 3. Trace once and list the kernel call sites

`extract_rebindables` walks the traced program (including inside `vmap`,
`scan`, `cond`, `remat`, `shard_map`, `custom_vjp` rules, ...) and returns one
`RebindableSite` per kernel call. Each site knows its kernel, its current
hyperparameters and the exact operand types (`in_avals_flat`) the kernel
sees, here already batched by `vmap`.

In [4]:
traced = model_jit.trace(params, xs)
sites = extract_rebindables(traced)
for s in sites:
  print(f"{s.name:18s} id={s.rebindable_id!r:12s} hp={s.rebindable.hyperparams}  "
        f"in={[str(a) for a in s.rebindable.in_avals_flat]}")

chunked_attention  id='attention'  hp={'block_q': 8}  in=['float32[4,512,64]', 'float32[4,512,64]', 'float32[4,512,64]']
chunked_mlp        id='mlp'        hp={'block_m': 8}  in=['float32[4,512,64]', 'float32[64,256]', 'float32[256,64]']
chunked_attention  id='attention'  hp={'block_q': 8}  in=['float32[4,512,64]', 'float32[4,512,64]', 'float32[4,512,64]']
chunked_mlp        id='mlp'        hp={'block_m': 8}  in=['float32[4,512,64]', 'float32[64,256]', 'float32[256,64]']
chunked_attention  id='attention'  hp={'block_q': 8}  in=['float32[4,512,64]', 'float32[4,512,64]', 'float32[4,512,64]']
chunked_mlp        id='mlp'        hp={'block_m': 8}  in=['float32[4,512,64]', 'float32[64,256]', 'float32[256,64]']


Several sites are identical (same kernel, same operand types): every layer has
the same shapes. A tuner only needs to benchmark each *distinct* site once.

In [5]:
def site_signature(site):
  return (site.name,
          tuple((a.shape, a.dtype) for a in site.rebindable.in_avals_flat))

distinct = {}
for s in sites:
  distinct.setdefault(site_signature(s), s)
print(f"{len(sites)} sites, {len(distinct)} distinct")

6 sites, 2 distinct


## 4. Benchmark each distinct site on its own

`site.call(*operands, **hyperparams)` runs just that kernel. The operands are
flat arrays typed like `site.rebindable.in_avals_flat`, so a tuner can
fabricate random inputs of the right types without ever seeing the real data
or the rest of the program. Candidates whose shapes don't divide evenly fail
to trace and are skipped.

In [6]:
def bench(f, *args, n=20):
  jax.block_until_ready(f(*args))  # compile + warm up
  ts = []
  for _ in range(n):
    t0 = time.perf_counter()
    jax.block_until_ready(f(*args))
    ts.append(time.perf_counter() - t0)
  return float(np.median(ts))

def random_operands(site, seed=0):
  keys = jax.random.split(jax.random.key(seed), len(site.rebindable.in_avals_flat))
  return [jax.random.normal(k, a.shape, a.dtype)
          for k, a in zip(keys, site.rebindable.in_avals_flat)]

search_space = {"attention": dict(block_q=[1, 4, 16, 64, 256, 512]),
                "mlp": dict(block_m=[1, 4, 16, 64, 256, 512])}

def tune_site(site, space, verbose=True):
  args = random_operands(site)
  results = {}
  for vals in itertools.product(*space.values()):
    hp = dict(zip(space, vals))
    try:
      results[vals] = bench(jax.jit(partial(site.call, **hp)), *args)
    except Exception as e:  # e.g. a shape that does not divide by the block
      results[vals] = float("inf")
    if verbose:
      print(f"  {site.name:18s} {hp}: {results[vals]*1e3:8.3f} ms")
  best = min(results, key=results.get)
  return dict(zip(space, best))

best_by_signature = {}
for sig, site in distinct.items():
  best_by_signature[sig] = tune_site(site, search_space[site.rebindable_id])
for (name, _), hp in best_by_signature.items():
  print(f"best {name}: {hp}")

  chunked_attention  {'block_q': 1}:   29.390 ms
  chunked_attention  {'block_q': 4}:   21.442 ms
  chunked_attention  {'block_q': 16}:   12.924 ms
  chunked_attention  {'block_q': 64}:   13.498 ms
  chunked_attention  {'block_q': 256}:   13.826 ms
  chunked_attention  {'block_q': 512}:   10.636 ms
  chunked_mlp        {'block_m': 1}:    9.016 ms
  chunked_mlp        {'block_m': 4}:    7.037 ms
  chunked_mlp        {'block_m': 16}:    6.882 ms
  chunked_mlp        {'block_m': 64}:    7.222 ms
  chunked_mlp        {'block_m': 256}:    6.930 ms
  chunked_mlp        {'block_m': 512}:    3.003 ms
best chunked_attention: {'block_q': 512}
best chunked_mlp: {'block_m': 512}


On CPU the largest block wins simply because `lax.map` turns each block into a
loop iteration, so fewer, bigger iterations are cheaper. On an accelerator the
trade-off (VMEM/shared-memory capacity, pipelining, occupancy) usually makes
the optimum shape dependent, which is exactly why tuning *per site*, with the
site's real shapes, matters.

## 5. Rebind the traced program and compare end to end

`rebind(traced, rule)` returns a new `Traced` with each site's hyperparameters
replaced. `rule` is a callable `site -> dict | None` (or a dict from site
`rebindable_id` to hyperparameters). It does not retrace the model: only the
rebound kernel bodies are retraced, each in its site's context.

In [7]:
tuned = rebind(traced, lambda site: best_by_signature.get(site_signature(site)))
for s in extract_rebindables(tuned):
  print(s.name, s.rebindable.hyperparams)

default_exe = traced.lower().compile()
tuned_exe = tuned.lower().compile()
np.testing.assert_allclose(default_exe(params, xs), tuned_exe(params, xs),
                           rtol=1e-4, atol=1e-4)
print(f"default: {bench(default_exe, params, xs)*1e3:.2f} ms")
print(f"tuned:   {bench(tuned_exe, params, xs)*1e3:.2f} ms")

chunked_attention {'block_q': 512}
chunked_mlp {'block_m': 512}
chunked_attention {'block_q': 512}
chunked_mlp {'block_m': 512}
chunked_attention {'block_q': 512}
chunked_mlp {'block_m': 512}
default: 78.45 ms
tuned:   61.62 ms


Rebinding is cheap enough to do on every candidate of the whole program too,
e.g. to validate a tuned table end to end or to tune jointly. Since the
hyperparameters are keyed, a table produced offline can also be applied
directly by id:

In [8]:
table = {"attention": dict(block_q=64), "mlp": dict(block_m=64)}
by_id = rebind(traced, table)
print([s.rebindable.hyperparams for s in extract_rebindables(by_id)][:2])

[{'block_q': 64}, {'block_m': 64}]


### Rebinding does not retrace the program

A quick check with a counter in a kernel body: tracing the model traces the
body once per call, as plain JAX would; `rebind` traces only each rebound
site's body, once; lowering traces no bodies.

In [9]:
n_traces = 0

@partial(rebindable, hyperparams="block", rebindable_id="scale")
def scaled(x, *, block=8):
  global n_traces
  n_traces += 1
  return 2.0 * x

f = jax.jit(lambda x: (scaled(scaled(x)), scaled(x[:4])))
tr = f.trace(jnp.ones(8))
print("after trace:  ", n_traces)
rb = rebind(tr, {"scale": dict(block=4)})
print("after rebind: ", n_traces)
rb.lower()
print("after lower:  ", n_traces)

after trace:   3
after rebind:  6
after lower:   6


## 6. Sharded program, tuned per shard on a single CPU

Kernels inside `shard_map` see **per-shard** operands. Here the model runs
with the batch *and* the sequence split over a 4-device mesh `(2, 2)`; the
site's `in_avals_flat` are therefore per-shard shapes, which is exactly what
we want to tune.

The sites below don't communicate (no collectives over the mesh axes), so we
can benchmark them **on one device** with plain `jax.jit`, using random
per-shard operands. A site that *does* communicate (e.g. calls
`lax.psum('data')`) has to be benchmarked under a `shard_map` over a mesh
with the same manual axis names, e.g. a small replica group; see
`RebindableSite.call`'s docstring.

In [10]:
# `make_mesh` defaults to `Explicit` axes: array types carry their sharding, so
# `shard_map` infers its `in_specs` (and `mesh`, from `jax.set_mesh`).
mesh = jax.make_mesh((2, 2), ("data", "seq"))
print(mesh.axis_types)

@partial(rebindable, hyperparams=("block_m",), rebindable_id="sharded_mlp")
def chunked_mlp_s(x, w1, w2, *, block_m=8):
  T, D = x.shape
  xb = x.reshape(T // block_m, block_m, D)
  return jax.lax.map(lambda xi: jax.nn.gelu(xi @ w1) @ w2, xb).reshape(T, -1)

@jax.jit
@jax.shard_map(out_specs=jax.P("data", "seq"))
def sharded_model(xs, w1, w2):  # per shard: xs is [B/2, T/2, D]
  Bl, Tl, Dl = xs.shape
  y = chunked_mlp_s(xs.reshape(Bl * Tl, Dl), w1, w2, block_m=8)  # local rows
  return xs + y.reshape(Bl, Tl, Dl)

Bs, Ts = 8, 1024
with jax.set_mesh(mesh):
  xs_big = jax.device_put(
      jax.random.normal(jax.random.key(2), (Bs, Ts, D), jnp.float32),
      jax.P("data", "seq"))
  w1, w2 = (jax.device_put(params[0][k], jax.P()) for k in ("w1", "w2"))
  print("x type:          ", jax.typeof(xs_big))
  traced_s = sharded_model.trace(xs_big, w1, w2)

[site_s] = extract_rebindables(traced_s)
print("site manual axes:", site_s.abstract_mesh.manual_axes)
print("per-shard in:    ", [str(a) for a in site_s.rebindable.in_avals_flat])

(Explicit, Explicit)
x type:           float32[8@data,1024@seq,64]
site manual axes: ('data', 'seq')
per-shard in:     ['float32[2048,64]{V:(data,seq)}', 'float32[64,256]', 'float32[256,64]']


The site sees `(Bs/2 * Ts/2, D) = (2048, 64)` rows, not the global
`(8 * 1024, 64)`. We now tune it on **one** CPU device, with per-shard-shaped
random operands placed there, and then rebind the 4-device program.

In [11]:
one_cpu = jax.devices()[0]
with jax.default_device(one_cpu):
  args = [jax.random.normal(jax.random.key(i), a.shape, a.dtype)
          for i, a in enumerate(site_s.rebindable.in_avals_flat)]
  print("tuning on:", {d for a in args for d in a.devices()})
  results = {}
  for bm in [1, 8, 64, 512, 2048]:
    results[bm] = bench(jax.jit(partial(site_s.call, block_m=bm)), *args)
    print(f"  block_m={bm:5d}: {results[bm]*1e3:8.3f} ms")
best_bm = min(results, key=results.get)
print("best:", best_bm)

tuning on: {CpuDevice(id=0)}
  block_m=    1:   10.056 ms
  block_m=    8:    6.758 ms
  block_m=   64:    6.574 ms
  block_m=  512:    3.708 ms
  block_m= 2048:    3.004 ms
best: 2048


Equivalently, a tuner can run the site under `shard_map` on a 1-device mesh
with the same axis names. This is the general recipe, and the one required
for sites that use collectives over those axes. Each shard then sees exactly
the per-shard types:

In [12]:
chip = jax.make_mesh((1, 1), ("data", "seq"), devices=jax.devices()[:1])

@jax.jit
@jax.shard_map(out_specs=jax.P())
def on_chip(*xs):
  return site_s.call(*xs, block_m=best_bm)

with jax.set_mesh(chip):
  chip_args = [jax.device_put(a, jax.P()) for a in args]
  print(on_chip(*chip_args).shape, f"{bench(on_chip, *chip_args)*1e3:.3f} ms")

(2048, 64) 2.318 ms


In [13]:
tuned_s = rebind(traced_s, {"sharded_mlp": dict(block_m=best_bm)})
with jax.set_mesh(mesh):
  default_s_exe, tuned_s_exe = traced_s.lower().compile(), tuned_s.lower().compile()
  out_default = default_s_exe(xs_big, w1, w2)
  out_tuned = tuned_s_exe(xs_big, w1, w2)
  np.testing.assert_allclose(out_default, out_tuned, rtol=1e-4, atol=1e-4)
  print(jax.typeof(out_tuned))
  print(f"4-device default: {bench(default_s_exe, xs_big, w1, w2)*1e3:.2f} ms")
  print(f"4-device tuned:   {bench(tuned_s_exe, xs_big, w1, w2)*1e3:.2f} ms")

float32[8@data,1024@seq,64]
4-device default: 11.00 ms
4-device tuned:   17.83 ms


## 7. Nesting rebindables

A rebindable's body may call other rebindables. The rules are simple:

* `extract_rebindables` lists a call **before** the calls nested in its body.
* `rebind` walks the program in that order. When the rule rebinds a call, its
  body is retraced (in the call's context) and the **same rule** is then
  applied to the calls in the new body. A parent's hyperparameter can thus
  change *which* nested calls exist, and one rule can choose both.
* Under `rebinding(rule)` the same happens when the program is lowered.

We build up to a parent that chooses between plain XLA and two *different*
rebindable kernels. First the two kernels: `matmul_rows` splits the rows of
`x` into blocks of `block_m`, `matmul_ksplit` accumulates over blocks of
`block_k` of the contracted dimension.

In [14]:
@partial(rebindable, hyperparams="block_m", rebindable_id="rows")
def matmul_rows(x, w, *, block_m=64):
  M, K = x.shape
  xb = x.reshape(M // block_m, block_m, K)
  return jax.lax.map(lambda xi: xi @ w, xb).reshape(M, -1)

@partial(rebindable, hyperparams="block_k", rebindable_id="ksplit")
def matmul_ksplit(x, w, *, block_k=64):
  (M, K), N = x.shape, w.shape[1]
  xs = x.reshape(M, K // block_k, block_k).transpose(1, 0, 2)
  ws = w.reshape(K // block_k, block_k, N)
  step = lambda acc, xw: (acc + xw[0] @ xw[1], None)
  return jax.lax.scan(step, jnp.zeros((M, N), x.dtype), (xs, ws))[0]

The parent's hyperparameter `impl` picks the implementation. All three
compute the same output type, as hyperparameters must not change it. The
program calls the parent twice, with different shapes.

In [15]:
@partial(rebindable, hyperparams="impl", rebindable_id="matmul")
def matmul(x, w, *, impl="xla"):
  if impl == "xla":
    return x @ w
  return {"rows": matmul_rows, "ksplit": matmul_ksplit}[impl](x, w)

def mlp_block(x, w1, w2):
  return matmul(jax.nn.gelu(matmul(x, w1)), w2)

ks = jax.random.split(jax.random.key(3), 3)
x7 = jax.random.normal(ks[0], (1024, 256), jnp.float32)
w7_1 = jax.random.normal(ks[1], (256, 512), jnp.float32) / 16
w7_2 = jax.random.normal(ks[2], (512, 256), jnp.float32) / 23
traced7 = jax.jit(mlp_block).trace(x7, w7_1, w7_2)

def show(traced):
  for s in extract_rebindables(traced):
    shapes = [a.shape for a in s.rebindable.in_avals_flat]
    print(f"  {s.rebindable_id:7s} {s.rebindable.hyperparams}  in={shapes}")

print("traced with impl='xla': no nested calls")
show(traced7)

traced with impl='xla': no nested calls
  matmul  {}  in=[(1024, 256), (256, 512)]
  matmul  {}  in=[(1024, 512), (512, 256)]


Rebinding the parents to `"rows"` retraces their bodies, so each now contains
a `matmul_rows` call, listed right after its parent. One rule (here a dict by
`rebindable_id`) can set the parent and the kernel it now uses in a single
pass, because the nested calls are rebound after their parent.

In [16]:
print("parents rebound to impl='rows':")
show(rebind(traced7, {"matmul": dict(impl="rows")}))
print("parents and the nested kernel, in one rule:")
show(rebind(traced7, {"matmul": dict(impl="ksplit"),
                      "ksplit": dict(block_k=128)}))

parents rebound to impl='rows':
  matmul  {'impl': 'rows'}  in=[(1024, 256), (256, 512)]
  rows    {}  in=[(1024, 256), (256, 512)]
  matmul  {'impl': 'rows'}  in=[(1024, 512), (512, 256)]
  rows    {}  in=[(1024, 512), (512, 256)]
parents and the nested kernel, in one rule:
  matmul  {'impl': 'ksplit'}  in=[(1024, 256), (256, 512)]
  ksplit  {'block_k': 128}  in=[(1024, 256), (256, 512)]
  matmul  {'impl': 'ksplit'}  in=[(1024, 512), (512, 256)]
  ksplit  {'block_k': 128}  in=[(1024, 512), (512, 256)]


### Tuning a choice between implementations

A tuner can search the parent's choice and, for each choice, tune whatever
nested kernels it uses: rebind the parents, extract the nested sites that
appear, tune each with `site.call` as before, then time the whole program
with the parent's choice and the tuned kernels. (A rule could also choose
`impl` per site, e.g. by operand types; here one choice is used throughout.)

In [17]:
nested_space = {"rows": dict(block_m=[16, 64, 256]),
                "ksplit": dict(block_k=[32, 128, 512])}
best_nested = {}  # site_signature -> hyperparameters

def rule_for(impl):
  def rule(site):
    if site.rebindable_id == "matmul":
      return dict(impl=impl)
    return best_nested.get(site_signature(site))
  return rule

end_to_end = {}
for impl in ["xla", "rows", "ksplit"]:
  for site in extract_rebindables(rebind(traced7, rule_for(impl))):
    sig = site_signature(site)
    if site.rebindable_id in nested_space and sig not in best_nested:
      best_nested[sig] = tune_site(site, nested_space[site.rebindable_id],
                                   verbose=False)
      print(f"  tuned {site.rebindable_id} {sig[1][0][0]}: {best_nested[sig]}")
  exe = rebind(traced7, rule_for(impl)).lower().compile()
  end_to_end[impl] = bench(exe, x7, w7_1, w7_2)
  print(f"impl={impl:7s} end to end: {end_to_end[impl]*1e3:.3f} ms")
best_impl = min(end_to_end, key=end_to_end.get)
print("best impl:", best_impl)

impl=xla     end to end: 6.098 ms
  tuned rows (1024, 256): {'block_m': 64}
  tuned rows (1024, 512): {'block_m': 256}
impl=rows    end to end: 20.318 ms
  tuned ksplit (1024, 256): {'block_k': 128}
  tuned ksplit (1024, 512): {'block_k': 512}
impl=ksplit  end to end: 28.231 ms
best impl: xla


On CPU, XLA's own matmul usually wins against these hand-written loops. That
is the point of keeping it as a choice: the parent lets a tuner fall back to
XLA where a custom kernel does not pay off, per program and per device.

Finally, a rule applies to a plain `jax.jit` call under `rebinding`: the
parents are rebound when the program is lowered, and the nested kernels in
their retraced bodies get the tuned values too.

In [18]:
mlp_jit = jax.jit(mlp_block)
reference = jax.nn.gelu(x7 @ w7_1) @ w7_2
for impl in ["xla", "rows", "ksplit"]:  # every choice computes the same thing
  with rebinding(rule_for(impl)):
    np.testing.assert_allclose(mlp_jit(x7, w7_1, w7_2), reference,
                               rtol=1e-4, atol=1e-4)
for impl in [best_impl, "rows"]:
  with rebinding(rule_for(impl)):
    t = bench(mlp_jit, x7, w7_1, w7_2)
  print(f"jit under rebinding(rule_for({impl!r})): {t*1e3:.3f} ms")

jit under rebinding(rule_for('xla')): 5.664 ms
jit under rebinding(rule_for('rows')): 16.727 ms


## Notes and limits

* Rebindables are **not differentiable** directly. For training, call them
  from the fwd/bwd rules of a `jax.custom_vjp`; forward and backward kernels
  are then separate sites, extracted from a program traced under `jax.grad`.
* Rebindables **may be nested**: rebinding a call retraces its body and
  applies the rule to the calls in it (parents first).
* Hyperparameters **must not change output types**; a rebind that does so
  fails at lowering.
* Inside a Pallas `fuser.fusible`, a rebindable receives the fused
  prologue/epilogue as `Fusion` operands, so `site.call` benchmarks the kernel
  *with* its fusions. Such sites must be benchmarked on a mesh of the
  same shape as the one they were traced under.